In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, base64, shutil, hashlib, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - write src/features_cic.py (shared by notebooks 11-13) and import.
# =============================================================================
(config.PROJECT_ROOT/'src'/'features_cic.py').write_bytes(base64.b64decode("IiIiQ0lDLUlEUzIwMTcgZmVhdHVyZSBlbmNvZGluZy4gU2luZ2xlIHNvdXJjZSBvZiB0cnV0aCBmb3IgQ0lDIGZlYXR1cmUgbWF0cmljZXMuIiIiCmltcG9ydCBudW1weSBhcyBucAoKTUVUQSA9IFsnbGFiZWxfcmF3JywgJ2xhYmVsJywgJ3N1YnR5cGUnLCAnZGF5JywgJ3NvdXJjZV9maWxlJywKICAgICAgICAnYXR0ZW1wdGVkX3JlbGFiZWxsZWRfYmVuaWduJywgJ3BhcnRpdGlvbiddCklEX0xJS0UgPSBbJ0Zsb3cgSUQnLCAnU3JjIElQJywgJ1NvdXJjZSBJUCcsICdEc3QgSVAnLCAnRGVzdGluYXRpb24gSVAnLAogICAgICAgICAgICdTcmMgUG9ydCcsICdTb3VyY2UgUG9ydCcsICdEc3QgUG9ydCcsICdEZXN0aW5hdGlvbiBQb3J0JywKICAgICAgICAgICAnUHJvdG9jb2wnLCAnVGltZXN0YW1wJ10KSU5URVJOQUwgPSBbJ19jYXB0dXJlX3RzJywgJ19vcmRlcicsICdfYmxvY2snXQoKZGVmIGZlYXR1cmVfY29scyhkZik6CiAgICBkcm9wID0gc2V0KE1FVEEpIHwgc2V0KElEX0xJS0UpIHwgc2V0KElOVEVSTkFMKQogICAgY29scyA9IFtjIGZvciBjIGluIGRmLmNvbHVtbnMgaWYgYyBub3QgaW4gZHJvcF0KICAgIHJldHVybiBkZltjb2xzXS5zZWxlY3RfZHR5cGVzKGluY2x1ZGU9W25wLm51bWJlcl0pLmNvbHVtbnMudG9saXN0KCkKCmRlZiBtYXRyaXgoZGYsIGNvbHMpOgogICAgWCA9IGRmW2NvbHNdLnRvX251bXB5KGR0eXBlPW5wLmZsb2F0NjQpCiAgICBYW35ucC5pc2Zpbml0ZShYKV0gPSBucC5uYW4KICAgIHJldHVybiBYCg=="))
import importlib
if 'features_cic' in sys.modules: importlib.reload(sys.modules['features_cic'])
import features_cic as fc

cic = pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed = cic[cic['day']=='wednesday'].reset_index(drop=True)
wed = wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)   # Heartbleed excluded (infeasible)

FCOLS = fc.feature_cols(wed)
SUB = config.SCOV_SUBSAMPLE_PER_SIDE
print('Wednesday DoS+Benign rows:', len(wed), '| feature columns:', len(FCOLS))
print('DoS variant totals:')
print(wed[wed.label=='DoS']['subtype'].value_counts().to_string())


Wednesday DoS+Benign rows: 477860 | feature columns: 81
DoS variant totals:
subtype
DoS Hulk            158449
DoS GoldenEye         7567
DoS Slowloris         3998
DoS Slowhttptest      1741


In [3]:
# =============================================================================
# Cell 3 - REALIZATIONS, recorded before any coverage (Amendment 9 A9.2/A9.3).
# Primary: 5 DoS variant-holdout realizations, Hulk always kept in calibration,
# held-out mass spanning 1,741 to 11,565. Secondary: within-Hulk covariate check.
# =============================================================================
V_HULK, V_GE = 'DoS Hulk', 'DoS GoldenEye'
V_SL, V_SH   = 'DoS Slowloris', 'DoS Slowhttptest'

PRIMARY_REALIZATIONS = {
    'R1_holdout_Slowhttptest':            [V_SH],
    'R2_holdout_Slowloris':               [V_SL],
    'R3_holdout_GoldenEye':               [V_GE],
    'R4_holdout_Slowloris_Slowhttptest':  [V_SL, V_SH],
    'R5_holdout_GoldenEye_Slowloris':     [V_GE, V_SL],
}
BENIGN_SPLIT_FRAC = 0.50          # benign split source/target, independent of variant (A9.2)

design = []
for name, H in PRIMARY_REALIZATIONS.items():
    kept = [v for v in [V_HULK, V_GE, V_SL, V_SH] if v not in H]
    held_mass = int(wed[wed.subtype.isin(H)].shape[0])
    kept_mass = int(wed[(wed.label=='DoS') & (wed.subtype.isin(kept))].shape[0])
    design.append({'realization':name,'held_out_variants':H,'kept_variants':kept,
                   'held_out_dos_mass':held_mass,'kept_dos_mass':kept_mass})
dz = pd.DataFrame(design)
print(dz[['realization','held_out_dos_mass','kept_dos_mass']].to_string(index=False))
assert (dz['kept_dos_mass'] > 1000).all(), 'a realization has too little kept DoS to calibrate'


                      realization  held_out_dos_mass  kept_dos_mass
          R1_holdout_Slowhttptest               1741         170014
             R2_holdout_Slowloris               3998         167757
             R3_holdout_GoldenEye               7567         164188
R4_holdout_Slowloris_Slowhttptest               5739         166016
   R5_holdout_GoldenEye_Slowloris              11565         160190


In [4]:
# =============================================================================
# Cell 4 - shift-measurement helpers (preregistration 9).
# Deterministic seeds (hashlib, not Python hash) for reproducibility.
# =============================================================================
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def seed_of(s):
    return int(hashlib.sha256(s.encode()).hexdigest(), 16) % (2**31)

def _sub(X, n, rng):
    return X[rng.choice(len(X), n, replace=False)] if len(X) > n else X

def s_cov_cv(Xc, Xe, seed, n=None, folds=5):
    n = SUB if n is None else n
    rng = np.random.default_rng(seed)
    Xc2, Xe2 = _sub(Xc, n, rng), _sub(Xe, n, rng)
    X = np.vstack([Xc2, Xe2]); y = np.r_[np.zeros(len(Xc2)), np.ones(len(Xe2))]
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed); a = []
    for tr, te in skf.split(X, y):
        clf = HistGradientBoostingClassifier(max_depth=4, max_iter=150, random_state=seed)
        clf.fit(X[tr], y[tr]); a.append(roc_auc_score(y[te], clf.predict_proba(X[te])[:,1]))
    return float(np.mean(a))

def s_cov_single(Xa, Xb, seed, n):
    rng = np.random.default_rng(seed)
    Xa2, Xb2 = _sub(Xa, n, rng), _sub(Xb, n, rng)
    X = np.vstack([Xa2, Xb2]); y = np.r_[np.zeros(len(Xa2)), np.ones(len(Xb2))]
    idx = rng.permutation(len(X)); cut = len(X)//2
    clf = HistGradientBoostingClassifier(max_depth=4, max_iter=150, random_state=seed)
    clf.fit(X[idx[:cut]], y[idx[:cut]])
    return float(roc_auc_score(y[idx[cut:]], clf.predict_proba(X[idx[cut:]])[:,1]))

def tv(pc, pe, classes):
    return float(0.5*sum(abs(pc.get(c,0.0)-pe.get(c,0.0)) for c in classes))

def priors(df, classes):
    v = df['label'].value_counts(normalize=True)
    return {c: float(v.get(c,0.0)) for c in classes}

CLASSES = ['Benign','DoS']
print('helpers ready; S_cov subsample per side =', SUB)


helpers ready; S_cov subsample per side = 20000


In [5]:
# =============================================================================
# Cell 5 - PERMUTATION NULL for S_cov (preregistration 9), computed once on the
# Wednesday pool. Single-split AUC per draw for tractability (logged in
# deviations.md). May take a few minutes.
# =============================================================================
NULL_SUB   = 8000
NULL_DRAWS = config.PERMUTATION_NULL_DRAWS

Xpool = fc.matrix(wed, FCOLS)
vals = []
for i in range(NULL_DRAWS):
    rng = np.random.default_rng(10_000 + i)
    idx = rng.permutation(len(Xpool)); half = len(idx)//2
    vals.append(s_cov_single(Xpool[idx[:half]], Xpool[idx[half:]], 20_000 + i, NULL_SUB))
    if (i+1) % 50 == 0: print(f'  null draw {i+1}/{NULL_DRAWS}')
null = np.array(vals)
NULL_Q95 = float(np.quantile(null, config.PERMUTATION_NULL_Q))
print(f'\npermutation null S_cov: median {np.median(null):.3f}, 95th pct {NULL_Q95:.3f}  (no-shift threshold)')


  null draw 50/200
  null draw 100/200
  null draw 150/200
  null draw 200/200

permutation null S_cov: median 0.500, 95th pct 0.511  (no-shift threshold)


In [ ]:
# =============================================================================
# Cell 6 - build each primary realization's partitions and measure shift.
# Source = kept-variant DoS + benign_source; target = held-out-variant DoS +
# benign_target. Five-partition split of the source (section 4).
# =============================================================================
def stratified_split(df, fractions, seed, col='label'):
    rng = np.random.default_rng(seed); names=list(fractions)
    fr=np.array([fractions[k] for k in names],float); big=names[int(np.argmax(fr))]
    asg=pd.Series(index=df.index,dtype=object)
    for _,sub in df.groupby(col,sort=True):
        idx=sub.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(fr*n).astype(int); c[names.index(big)]+=n-c.sum(); s=0
        for nm,k in zip(names,c): asg.loc[idx[s:s+k]]=nm; s+=k
    return asg

config.PROC_DIR.mkdir(parents=True, exist_ok=True)
benign = wed[wed.label=='Benign']
rng_b  = np.random.default_rng(777)
b_perm = benign.index.to_numpy().copy(); rng_b.shuffle(b_perm)
b_cut  = int(len(b_perm)*BENIGN_SPLIT_FRAC)
BEN_SRC, BEN_TGT = set(b_perm[:b_cut]), set(b_perm[b_cut:])

rows=[]
for r in design:
    name, H, kept = r['realization'], r['held_out_variants'], r['kept_variants']
    dos_src = wed[(wed.label=='DoS') & (wed.subtype.isin(kept))]
    dos_tgt = wed[(wed.label=='DoS') & (wed.subtype.isin(H))]
    src = pd.concat([dos_src, benign.loc[sorted(BEN_SRC)]]).copy()
    tgt = pd.concat([dos_tgt, benign.loc[sorted(BEN_TGT)]]).copy()
    src = src.assign(partition=stratified_split(src, config.SPLIT_FRACTIONS, seed_of(name)).values)
    S_pool = src[src.partition=='source_cal_pool']

    np.save(config.PROC_DIR/f'cic_{name}_srcpool_idx.npy', np.sort(S_pool.index.to_numpy()))
    np.save(config.PROC_DIR/f'cic_{name}_target_idx.npy',  np.sort(tgt.index.to_numpy()))

    Xc = fc.matrix(S_pool, FCOLS); Xe = fc.matrix(tgt, FCOLS)
    scov = s_cov_cv(Xc, Xe, seed=seed_of(name))
    slab = tv(priors(S_pool,CLASSES), priors(tgt,CLASSES), CLASSES)
    ssup = float(tgt.subtype.isin(H).mean())
    rows.append({'realization':name,'held_out_dos_mass':r['held_out_dos_mass'],
                 'n_src_pool':len(S_pool),'n_target':len(tgt),
                 'S_cov':round(scov,4),'S_cov_above_null':bool(scov>NULL_Q95),
                 'S_lab':round(slab,4),'S_sup':round(ssup,4)})
    print(f"{name:38s} S_cov={scov:.3f} S_lab={slab:.3f} S_sup={ssup:.3f}")

shift = pd.DataFrame(rows); shift['null_q95']=round(NULL_Q95,4)
shift.to_csv(config.REPORTS_DIR/'ladder_shift_measures_cicids2017.csv', index=False)
print('\n', shift.to_string(index=False))


R1_holdout_Slowhttptest                S_cov=0.769 S_lab=0.515 S_sup=0.011
R2_holdout_Slowloris                   S_cov=0.760 S_lab=0.497 S_sup=0.025
R3_holdout_GoldenEye                   S_cov=0.772 S_lab=0.470 S_sup=0.047
R4_holdout_Slowloris_Slowhttptest      S_cov=0.764 S_lab=0.484 S_sup=0.036


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



R5_holdout_GoldenEye_Slowloris         S_cov=0.773 S_lab=0.441 S_sup=0.070
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1141/1217264234.py", line 47, in <cell line: 0>
    shift.to_csv(config.REPORTS_DIR/'ladder_shift_measures_cicids2017.csv', index=False)
  File "/usr/local/lib/python3.12/dist-packages/pandas/util/_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/generic.py", line 3967, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py", line 1014, in to_csv
    csv_formatter.save()
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/formats/csvs.py", line 251, in sav

In [ ]:
# =============================================================================
# Cell 7 - SECONDARY within-Hulk covariate check (Amendment 9 A9.3).
# DoS Hulk only, split earlier vs later by flow order at fixed variant support.
# =============================================================================
hulk = wed[wed.subtype==V_HULK].reset_index(drop=True)
ho = np.arange(len(hulk)); cut = len(ho)//2
Xa = fc.matrix(hulk.iloc[ho[:cut]], FCOLS); Xb = fc.matrix(hulk.iloc[ho[cut:]], FCOLS)
scov_hulk = s_cov_cv(Xa, Xb, seed=4242)
verdict = 'above null (covariate shift present)' if scov_hulk > NULL_Q95 else 'below null (weak within-attack covariate drift)'
hcheck = {'check':'within_Hulk_covariate','n_hulk':int(len(hulk)),
          'S_cov':round(scov_hulk,4),'null_q95':round(NULL_Q95,4),'verdict':verdict}
(config.REPORTS_DIR/'cicids2017_hulk_covariate_check.json').write_text(json.dumps(hcheck,indent=2))
print(json.dumps(hcheck, indent=2))


In [ ]:
# =============================================================================
# Cell 8 - ladder manifest, deviation log, commit.
# =============================================================================
man = {'dataset':'cicids2017','environment':'wednesday_within_day',
       'primary_ladder':'DoS variant holdout (support shift), Amendment 9 A9.2',
       'secondary_check':'within-Hulk covariate, Amendment 9 A9.3',
       'benign_split_frac':BENIGN_SPLIT_FRAC,'scov_subsample_per_side':SUB,
       'null_subsample_per_side':NULL_SUB,'null_draws':NULL_DRAWS,'null_q95':round(NULL_Q95,4),
       'realizations':{r['realization']:{'held_out_variants':r['held_out_variants'],
                                         'held_out_dos_mass':r['held_out_dos_mass']} for r in design}}
(config.REPORTS_DIR/'ladder_manifest_cicids2017.json').write_text(json.dumps(man,indent=2))

dev = config.REPORTS_DIR/'deviations.md'
note = ('\n## nb11 - permutation null S_cov uses single-split AUC at 8k/side for tractability '
        '(measured S_cov is 5-fold at 20k/side per section 9). Logged before any coverage.\n')
if dev.exists() and 'nb11 - permutation null' not in dev.read_text():
    with open(dev,'a') as f: f.write(note)

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,d in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,d)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb11: CIC DoS variant-holdout ladder, within-Hulk covariate check, shift measures')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-4',show=False).stdout)
